### Lab 3.1: Basic Neural Network in PyTorch - Solution

Let's create a linear classifier one more time, but using PyTorch's automatic differentiation and optimization algorithms.  Then you will extend the perceptron into a multi-layer perceptron (MLP).

In [18]:
import numpy as np
import torch

We need to explicitly tell PyTorch when creating a tensor that we are interested in later computing its gradient

In [19]:
a = torch.tensor(5.,requires_grad=True)
a

tensor(5., requires_grad=True)

In [20]:
b = torch.tensor(6.,requires_grad=True)
c = 2*a+3*b
c

tensor(28., grad_fn=<AddBackward0>)

To extract the gradients, we first need to call `backward()`.

In [21]:
c.backward()

Now to get the gradient of any variable with respect to `c`, we simply access the `grad` attribute of that variable.

In [22]:
a.grad

tensor(2.)

In [23]:
b.grad

tensor(3.)

Let's load and format the Palmer penguins dataset for multi-class classification.

In [24]:
from palmerpenguins import load_penguins
from matplotlib import pyplot as plt

In [25]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert species labels to integers
y = df['species'].map({'Adelie':0,'Chinstrap':1,'Gentoo':2}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [26]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [27]:
X = torch.tensor(X).float()
y = torch.tensor(y).long()

In [28]:
from torch import nn

The `torch.nn.Sequential` class creates a feed-forward network from a list of `nn.Module` objects.  Here we provide a single `nn.Linear` class which performs an affine transformation ($Wx+b$) so that we will have a linear classifier.

In [29]:
linear_model = torch.nn.Sequential(
    torch.nn.Linear(2,3), # two inputs, three outputs
)

Now we create a cross-entropy loss function object and a stochastic gradient descent (SGD) optimizer.

In [30]:
loss_fn = torch.nn.CrossEntropyLoss()

In [31]:
lr = 1e-2
opt = torch.optim.SGD(linear_model.parameters(), lr=lr)

Finally we can iteratively optimize the model.

In [32]:
epochs = 100
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = linear_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item()}')

epoch 0: loss is 3.6084673404693604
epoch 1: loss is 2.5213570594787598
epoch 2: loss is 1.5691739320755005
epoch 3: loss is 0.7818202972412109
epoch 4: loss is 0.3773713707923889
epoch 5: loss is 0.29090431332588196
epoch 6: loss is 0.2596416771411896
epoch 7: loss is 0.24226446449756622
epoch 8: loss is 0.23078148066997528
epoch 9: loss is 0.22243767976760864
epoch 10: loss is 0.21599434316158295
epoch 11: loss is 0.21080133318901062
epoch 12: loss is 0.20648081600666046
epoch 13: loss is 0.20279672741889954
epoch 14: loss is 0.19959326088428497
epoch 15: loss is 0.196763277053833
epoch 16: loss is 0.19423054158687592
epoch 17: loss is 0.19193917512893677
epoch 18: loss is 0.18984739482402802
epoch 19: loss is 0.18792322278022766
epoch 20: loss is 0.1861417591571808
epoch 21: loss is 0.1844833493232727
epoch 22: loss is 0.18293215334415436
epoch 23: loss is 0.18147538602352142
epoch 24: loss is 0.18010246753692627
epoch 25: loss is 0.17880459129810333
epoch 26: loss is 0.177574411034

### Exercises

Extend the above code to implement an MLP with a single hidden layer of size 100.

Write code to compute the accuracy of each model.

Can you get the MLP to outperform the linear model?

In [47]:
nonlinear_model = torch.nn.Sequential(
    torch.nn.Linear(2,100),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(100,3),
)

lr = 0.01
opt = torch.optim.SGD(nonlinear_model.parameters(), lr=lr)

epochs = 1000
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = nonlinear_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item()}')


def accuracy(model, X, y):
    with torch.no_grad():
        z = model(X)
        preds = torch.argmax(z, dim=1)
    correct = (preds == y).sum().item()
    return correct / len(y)

print(f"Linear model accuracy is: {accuracy(linear_model, X, y)}")
print(f"Nonlinear model accuracy is: {accuracy(nonlinear_model, X, y)}")

epoch 0: loss is 3.4916725158691406
epoch 1: loss is 1.092286467552185
epoch 2: loss is 0.7445605993270874
epoch 3: loss is 0.4853847324848175
epoch 4: loss is 0.3893562853336334
epoch 5: loss is 0.33267319202423096
epoch 6: loss is 0.2894444167613983
epoch 7: loss is 0.2562578320503235
epoch 8: loss is 0.23095546662807465
epoch 9: loss is 0.2121390700340271
epoch 10: loss is 0.19854062795639038
epoch 11: loss is 0.18891334533691406
epoch 12: loss is 0.18209850788116455
epoch 13: loss is 0.1771279126405716
epoch 14: loss is 0.17329677939414978
epoch 15: loss is 0.1701590120792389
epoch 16: loss is 0.16746757924556732
epoch 17: loss is 0.165092334151268
epoch 18: loss is 0.16296181082725525
epoch 19: loss is 0.1610317826271057
epoch 20: loss is 0.1592714637517929
epoch 21: loss is 0.15765748918056488
epoch 22: loss is 0.15617097914218903
epoch 23: loss is 0.15479663014411926
epoch 24: loss is 0.1535215675830841
epoch 25: loss is 0.1523348093032837
epoch 26: loss is 0.15122705698013306
e